# 3_6 — Абляции DPI-Flow (P1, component-contribution)

Отдельный ноутбук абляций. Каждая 1:1 отключает один компонент вклада; обучение/оценка на объектном (leakage-free) фолде — тот же протокол, что P0. Метрики **по 3 состояниям** (liq/stab/nostab + worst).

Варианты: full, wo_conformal, gaussian_posterior, wo_ode, wo_monotone, wo_risk_softauc, wo_censored_nliq, miss_vs, miss_grainsize. Логика — в `liquefaction_ai.evaluation.ablation_study`.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')):
    REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src'))
os.chdir(REPO)
import numpy as np, pandas as pd
TABLES = os.path.join(REPO, 'results', 'analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB = os.path.join(REPO, 'results', 'tables'); os.makedirs(PUB, exist_ok=True)
FIGS = os.path.join(REPO, 'results', 'analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nikita/Desktop/projects/liquefaction-ai


In [2]:
import json, torch
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.ablation_study import run_ablation_fold, aggregate_ablations
device = torch.device('cpu'); QUICK = False
pop, config = load_population_artifact('data/dataset')
meta = pop['meta']
feat_names = json.load(open('data/dataset/feature_names.json'))['static_feature_names']
from liquefaction_ai.evaluation.cross_validation import build_folds
folds = build_folds(meta, config, seed=42, loo=False, n_splits=5)
print('folds:', len(folds))

folds: 5


## Прогон абляций (по умолчанию фолд 0; для CI — по всем фолдам)

In [3]:
FOLDS = []   # [] → все 5 фолдов (n_splits=5); центральный architectural claim — по всем фолдам
use_folds = FOLDS if FOLDS else range(len(folds))
rows = []
for k in use_folds:
    rows.append(run_ablation_fold(pop, config, folds[k], k, feat_names, device, quick=QUICK))
abl = pd.concat(rows, ignore_index=True)
abl.to_csv(os.path.join(TABLES, 'ablations.csv'), index=False)
abl[['fold','ablation','AUPRC','ECE','Coverage_90','Traj_RMSE_worst','Physics_Violation_Rate','N_liq_logMAE']].round(4)

,fold,ablation,AUPRC,ECE,Coverage_90,Traj_RMSE_worst,Physics_Violation_Rate,N_liq_logMAE
0,0,full,1.0000,0.0243,0.7779,0.3069,0.0000,0.0846
1,0,wo_conformal,1.0000,0.0243,0.6561,0.3069,0.0000,0.0846
2,0,gaussian_posterior,0.9999,0.0102,0.6952,0.3057,0.0000,0.1387
3,0,wo_ode,1.0000,0.0118,0.8483,0.0897,0.0000,0.5797
4,0,blackbox_raw,1.0000,0.0006,0.7534,0.0816,0.6429,0.3354
5,0,wo_monotone,1.0000,0.0243,0.7779,0.3069,0.0000,0.0846
6,0,wo_risk_softauc,0.9993,0.0264,0.7410,0.3174,0.0000,0.0785
7,0,wo_censored_nliq,1.0000,0.0030,0.7388,0.3067,0.0000,0.0714
8,0,miss_vs,1.0000,0.0078,0.7587,0.3096,0.0000,0.1255
9,0,miss_grainsize,1.0000,0.0004,0.6977,0.3120,0.0000,0.0655


In [4]:
summary = aggregate_ablations(abl)
summary.to_csv(os.path.join(TABLES, 'ablations_summary.csv'), index=False)
summary[[c for c in ['ablation','n_folds','AUPRC_mean','ECE_mean','Traj_RMSE_worst_mean','Physics_Violation_Rate_mean','N_liq_logMAE_mean'] if c in summary.columns]]

,ablation,n_folds,AUPRC_mean,ECE_mean,Traj_RMSE_worst_mean,Physics_Violation_Rate_mean,N_liq_logMAE_mean
0,blackbox_raw,5,0.9990,0.0409,0.1155,0.7468,0.9889
1,full,5,0.9987,0.0449,0.2631,0.0295,0.3900
2,gaussian_posterior,5,0.9983,0.0492,0.2644,0.0100,0.3758
3,miss_grainsize,5,0.9991,0.0353,0.2642,0.0109,0.4113
4,miss_vs,5,0.9987,0.0369,0.2623,0.0261,0.4178
5,no_aux,5,0.9989,0.0385,0.2620,0.0318,0.4655
6,no_prefix,5,0.9964,0.0468,0.2968,0.0440,0.4409
7,wo_censored_nliq,5,0.9987,0.0268,0.2618,0.1070,0.1865
8,wo_conformal,5,0.9987,0.0449,0.2631,0.0295,0.3900
9,wo_monotone,5,0.9987,0.0449,0.2631,0.0295,0.3900


## Ablation figures (component contribution, English)

In [5]:
from liquefaction_ai.evaluation import ablation_bars
from IPython.display import display
display(ablation_bars(summary, metric='Traj_RMSE_worst', higher_better=False, out_dir=FIGS))
display(ablation_bars(summary, metric='AUPRC', higher_better=True, out_dir=FIGS))

<Figure size 720x740 with 1 Axes>

<Figure size 720x740 with 1 Axes>

## Prefix-length sensitivity (п.6)

Ре-материализуйте `data/dataset` с разным `config.prefix_len` (ноутбуки 1_x), затем для каждого K запустите этот ноутбук с `only='full'` и сравните метрики онсета/траектории vs длина префикса.